# 🎧 Audiobook Generation System - Colab Runner

In [ ]:
# [1] Clone repo
import os
REPO = "CSC15012-Final-Project-Audiobook-Generation-System"
if not os.path.exists(REPO):
    !git clone https://github.com/TiiAyyLuvBear/{REPO}
%cd {REPO}
print(os.getcwd())

In [ ]:
# [2] Cài thư viện
!pip install streamlit -q
!pip install -r requirements.txt -q 2>/dev/null || echo 'requirements.txt skipped'
!pip install -r streamlit_requirements.txt -q 2>/dev/null || echo 'streamlit_requirements.txt skipped'
print('✅ Done')

In [ ]:
# [3] Cài cloudflared
!wget -qO /usr/local/bin/cloudflared \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [13]:
# [4] Chạy Streamlit + tunnel
import subprocess, threading, time, socket, re, sys

PORT = 8501
_log = []
_url = [None]

# Kill process cũ đang chiếm port (fix lỗi 'Port 8501 is not available')
print(f'🔪 Giải phóng port {PORT}...')
subprocess.run(f'fuser -k {PORT}/tcp 2>/dev/null || true', shell=True)
time.sleep(2)
print('✅ Port sẵn sàng')

# --- Streamlit ---
def _start_streamlit():
    p = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'streamlit_app.py',
         f'--server.port={PORT}',
         '--server.headless=true',
         '--server.enableCORS=false',
         '--server.enableXsrfProtection=false'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        _log.append(line.rstrip())

threading.Thread(target=_start_streamlit, daemon=True).start()

def _ready(port):
    try:
        socket.create_connection(('127.0.0.1', port), timeout=1).close()
        return True
    except OSError:
        return False

print('⏳ Chờ Streamlit', end='')
ok = False
for _ in range(30):
    time.sleep(1); print('.', end='', flush=True)
    if _ready(PORT): ok = True; break
print()

if not ok:
    print('⚠️  Streamlit chưa lên — xem log tại Cell [5]')
else:
    print(f'✅ Streamlit OK tại port {PORT}')

# --- Tunnel ---
def _start_tunnel():
    p = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m: _url[0] = m.group(); break

threading.Thread(target=_start_tunnel, daemon=True).start()

print('🌐 Tạo tunnel', end='')
for _ in range(30):
    time.sleep(1); print('.', end='', flush=True)
    if _url[0]: break
print()

if _url[0]:
    print('=' * 60)
    print(f'🚀  {_url[0]}')
    print('=' * 60)
else:
    print('❌ Không lấy được URL — xem log tại Cell [5]')

🔪 Giải phóng port 8501...
✅ Port sẵn sàng
⏳ Chờ Streamlit....
✅ Streamlit OK tại port 8501
🌐 Tạo tunnel......
🚀  https://tobacco-they-breeding-fresh.trycloudflare.com


In [17]:
# [5] Xem log Streamlit bất cứ lúc nào
print(f'=== Streamlit Log ({len(_log)} dòng) ===')
print('\n'.join(_log) or '(chưa có log)')

=== Streamlit Log (22 dòng) ===


2026-05-06 05:09:46.341 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.30.0.2:8501
  External URL: http://34.87.184.242:8501

`torch_dtype` is deprecated! Use `dtype` instead!
2026-05-06 05:10:50.020719: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778044250.060078   25999 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778044250.070810   25999 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778044250.106142   25999 computation_placer.cc:177] computation placer already registered. Please check 